# Bài 13: Cơ chế chú ý và Transformer
**Mở hộp đen AI · Tập 2 · Phòng thí nghiệm Colab**

Ô 1 đến ô 6 là phần lõi, chỉ dùng NumPy. Ô 7 là phần **mở rộng, kết quả tuỳ máy**. Các câu trong bài do nhóm tác giả tự viết.
Ebook Bên trong cỗ máy Trí tuệ nhân tạo - Tập 2. Phát hành tại https://montinhoc.vn

## Chặng 1: Chạy
### Ô 1. Softmax: biến điểm số thành tỉ lệ chú ý

In [ ]:
import numpy as np

def softmax(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))   # chống tràn số
    return e / e.sum(axis=-1, keepdims=True)

print(softmax(np.array([1.0, 0.0, 0.0])).round(3))
print(softmax(np.array([2.0, 1.0, 0.0])).round(3))
print(softmax(np.array([20.0, 10.0, 0.0])).round(3))

### Ô 2. Tính chú ý cho 3 từ: “chó”, “xương”, “nó”
Mỗi từ có 4 đặc trưng: [sinh vật, đồ ăn, từ thay thế, hành động].

In [ ]:
cho   = np.array([1, 0, 0, 0])
xuong = np.array([0, 1, 0, 0])
no    = np.array([0, 0, 1, 0])
K = np.array([cho, xuong, no])        # khoá: mỗi từ "giới thiệu" mình
V = K.copy()                          # giá trị: điều mỗi từ "đưa ra"
q = np.array([2, 0, 0, 0])            # truy vấn của "nó": tìm một sinh vật

diem = K @ q / np.sqrt(len(q))        # tích vô hướng, chia căn của 4
trong_so = softmax(diem)
print("Điểm:     ", diem)
print("Chú ý:    ", trong_so.round(3))
print("Đầu ra 'nó':", (trong_so @ V).round(3))

### Ô 3. Cả một câu: “nó” nhìn về đâu?

In [ ]:
#            sinh vật  đồ ăn  thay thế  hành động  nối  trạng thái
DAC_TRUNG = {"con":   [0, 0, 0, 0, 0, 0], "chó":   [1, 0, 0, 0, 0, 0],
             "gặm":   [0, 0, 0, 1, 0, 0], "khúc":  [0, 0, 0, 0, 0, 0],
             "xương": [0, 1, 0, 0, 0, 0], "vì":    [0, 0, 0, 0, 1, 0],
             "nó":    [0, 0, 1, 0, 0, 0], "đói":   [0, 0, 0, 0, 0, 1]}
cau = "con chó gặm khúc xương vì nó đói".split()
X = np.array([DAC_TRUNG[t] for t in cau], dtype=float)

Wq = np.zeros((6, 6)); Wq[2, 0] = 4    # thay thế -> tìm sinh vật
Wk = np.eye(6)                        # khoá: giữ nguyên đặc trưng
Wv = np.eye(6)

def chu_y(X, Wq, Wk, Wv):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    A = softmax(Q @ K.T / np.sqrt(K.shape[1]))   # bảng chú ý n x n
    return A, A @ V

A, Y = chu_y(X, Wq, Wk, Wv)
i = cau.index("nó")
for t, a in zip(cau, A[i]):
    print(f"nó -> {t:6}: {a:.2f}")

## Chặng 2: Đọc
1. Ở ô 1, khi các điểm cách nhau xa (20, 10, 0), tỉ lệ chú ý thay đổi thế nào?
2. Ở ô 2, vì sao “nó” chú ý vào “chó” nhiều nhất? Con số 0,576 đến từ đâu?
3. Ở ô 3, những từ có truy vấn bằng 0 (như “gặm”) sẽ chú ý thế nào? Thử in `A[2]`.

## Chặng 3: Sửa
### Ô 4. Chú ý không biết thứ tự từ

In [ ]:
TU = {"chó": [1, 0, 0], "cắn": [0, 1, 0], "mèo": [0, 0, 1]}
def tu_chu_y(cau):                    # Q = K = V = vectơ của từ
    X = np.array([TU[t] for t in cau.split()], dtype=float)
    return chu_y(X, np.eye(3), np.eye(3), np.eye(3))[1]

Y1, Y2 = tu_chu_y("chó cắn mèo"), tu_chu_y("mèo cắn chó")
print("Đầu ra của 'cắn' giống nhau? ", np.allclose(Y1[1], Y2[1]))

def vi_tri(n, d=3):                   # mỗi vị trí một vectơ riêng
    return np.array([[np.sin((p + 1) * (k + 1)) for k in range(d)] for p in range(n)])

def tu_chu_y_vi_tri(cau):
    X = np.array([TU[t] for t in cau.split()], dtype=float) + vi_tri(3)
    return chu_y(X, np.eye(3), np.eye(3), np.eye(3))[1]

Y1, Y2 = tu_chu_y_vi_tri("chó cắn mèo"), tu_chu_y_vi_tri("mèo cắn chó")
print("Thêm vị trí, còn giống nhau?", np.allclose(Y1[1], Y2[1]))

### Ô 5. Hai đầu chú ý: một đầu tìm sinh vật, một đầu nhìn từ ngay trước

In [ ]:
n = len(cau)
P = np.eye(n)                          # vị trí dạng một nóng
X2 = np.hstack([X, P])                 # đặc trưng + vị trí: 6 + 8 = 14 cột

Wq1 = np.zeros((14, 14)); Wq1[2, 0] = 8                 # đầu 1: như ô 3, mạnh hơn
Wq2 = np.zeros((14, 14))
for p in range(1, n):
    Wq2[6 + p, 6 + p - 1] = 8                           # đầu 2: hỏi "vị trí ngay trước"
A1, _ = chu_y(X2, Wq1, np.eye(14), np.eye(14))
A2, _ = chu_y(X2, Wq2, np.eye(14), np.eye(14))
print("Đầu 1, 'nó' nhìn nhiều nhất:", cau[A1[i].argmax()])
print("Đầu 2, mỗi từ nhìn nhiều nhất:")
print([f"{cau[r]}->{cau[A2[r].argmax()]}" for r in range(1, n)])

### Ô 6. Câu càng dài, bảng chú ý càng lớn

In [ ]:
for n in [8, 1_000, 100_000]:
    so_o = f"{n * n:,}".replace(",", ".")          # dấu chấm ngăn hàng nghìn
    print(f"{n:>7} token -> {so_o:>15} ô trong bảng chú ý")

## Mở rộng: kết quả tuỳ máy
### Ô 7. Hỏi PhoBERT điền từ còn thiếu
Cần Internet. PhoBERT đọc câu đã tách từ, các tiếng của một từ nối bằng dấu `_`.

In [ ]:
!pip install -q transformers
from transformers import pipeline
dien_tu = pipeline("fill-mask", model="vinai/phobert-base")
for kq in dien_tu("Hà_Nội là <mask> của Việt_Nam ."):
    print(kq["token_str"], round(kq["score"], 3))

## Chặng 4: Tự làm
1. Ở ô 3, đổi `đói` thành `ngon`. Theo nghĩa, “nó” bây giờ chỉ gì? Máy của em còn đúng không?
2. Ở ô 3, sửa `Wq` để “nó” đi tìm **đồ ăn** thay vì sinh vật.
3. Ở ô 2, đổi truy vấn thành `[8, 0, 0, 0]`. Chú ý vào “chó” tăng lên bao nhiêu?

**Nhật ký dùng AI:** Em có nhờ AI ở bước nào không? Nó sai ở đâu? Em sửa thế nào?

*(ghi vào đây)*